In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.library.restartPython()

In [0]:
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
log_dir = "/Workspace/Users/saythu000@gmail.com/provider/logs"
dbutils.fs.mkdirs(f"file:{log_dir}")

# Create log filename with timestamp
log_filename = f"{log_dir}/provider_person_bridge_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Provider Person Bridge Processing Started")
logger.info(f"Log file: {log_filename}")
logger.info("=" * 60)

In [0]:
try:
    dbutils.widgets.text("ClientContainer", "claimsprocessing", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "claimsprocessing"

# Wrap catalog name in backticks for safety (e.g. `274`)
safe_catalog = f"`{client_container}`"

sourcePath = f"/Volumes/{client_container}/bronze/processed_parquet/provider"
silverTable = f"{safe_catalog}.silver.provider_person_bridge"

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Table: {silverTable}")


In [0]:
%pip install recordlinkage -q

In [0]:
from pyspark.sql.functions import date_format, monotonically_increasing_id, sha2, concat_ws, col, row_number, substring, lower, coalesce, upper, trim, regexp_replace
from pyspark.sql.window import Window
import pandas as pd
import recordlinkage
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [0]:
# COMMAND ----------

# Databricks notebook source
# DBTITLE 1,Helper Functions
def table_exists(tableToCheck):
    """Check if a table or path exists and can be read"""
    try:
        if '/' in tableToCheck:
            spark.read.format("parquet").load(tableToCheck)
        else:
            # Adding .schema forces Spark to validate table existence immediately
            spark.table(tableToCheck).schema
        logger.info(f"Data check: {tableToCheck} - Data exists")
        return True
    except Exception as e:
        logger.warning(f"Data check: {tableToCheck} - Data does not exist or is inaccessible: {str(e)}")
        return False

In [0]:
def LoadSource(sourcePath):
    logger.info(f"Starting LoadSource from: {sourcePath}")
    dfMBPGold = spark.read.format("parquet").load(sourcePath)
    logger.info(f"Raw records loaded: {dfMBPGold.count()}")
    windowPartition = Window.partitionBy(col("FILE_ID")).orderBy(col("RecordHash").desc())
    sparkMem_df = dfMBPGold.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfMBPGold.columns), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FILE_ID"), col("RowNumber"))) \
        .withColumn("NPI", upper(trim(col("NPI")))) \
        .withColumn("NPIFormatted", regexp_replace(trim(col("NPI")), "[^0-9]", "")) \
        .withColumn("DEA", upper(trim(col("DEA")))) \
        .withColumn("DEAFormatted", regexp_replace(trim(col("DEA")), "[^a-zA-Z0-9]", "")) \
        .withColumn("ProviderID", upper(trim(col("ProviderID")))) \
        .withColumn("LastName", upper(trim(col("LastName")))) \
        .withColumn("FirstName", upper(trim(col("FirstName")))) \
        .withColumn("LastInitial", upper(substring(trim(col("LastName")), 1, 1))) \
        .withColumn("FirstInitial", upper(substring(trim(col("FirstName")), 1, 1))) \
        .withColumn("PayorID", upper(trim(col("PayorID")))) \
        .withColumn("FileLayoutID", col("FILE_LAYOUT_ID").cast("string")) \
        .withColumn("FileID", col("FILE_ID")) \
        .select("UniqueRecord", "FileLayoutID", "FileID", "RowNumber", "LastName", "FirstName", "NPI", "DEA", "PayorID", "ProviderID", "LastInitial", "FirstInitial", "NPIFormatted", "DEAFormatted")
    logger.info(f"Processed records after transformations: {sparkMem_df.count()}")
    logger.info("LoadSource completed successfully")
    return sparkMem_df

In [0]:
RuleNPIColumns = ["NPIFormatted", "FirstInitial", "LastInitial", 3]
RuleDEAColumns = ["DEAFormatted", "FirstInitial", "LastInitial", 3]
RuleProvIDColumns = ["ProviderID", "FirstInitial", "LastInitial", 3]
RuleDemographicsColumns = ["LastName", "FirstName", 2]
RulesAll = [RuleNPIColumns, RuleDEAColumns, RuleProvIDColumns, RuleDemographicsColumns]
CompareColumns = ["LastInitial", "FirstInitial", "LastName", "FirstName", "NPIFormatted", "DEAFormatted", "ProviderID"]

In [0]:
def RulesToCompare(rules, pandasMem_df):
    logger.info("Starting RulesToCompare function")
    matchesAllRules_df = pd.DataFrame()
    for i, lst in enumerate(rules, 1):
        # Skip rules if matching column is empty or not filled in the df
        col_to_block = lst[0]
        if pandasMem_df[col_to_block].dropna().empty or (pandasMem_df[col_to_block] == '').all():
            logger.info(f"Rule {i} ({col_to_block}): Skipping because matching column is empty")
            continue
        
        indexer = recordlinkage.Index()
        indexer.block(col_to_block)
        candidatesBlock = indexer.index(pandasMem_df)
        logger.info(f"Rule {i} ({col_to_block}): Found {len(candidatesBlock)} candidate pairs")
        
        compareBlock = recordlinkage.Compare()
        threshold = lst[-1]
        for col in CompareColumns:
            compareBlock.exact(col, col, label=str(col))
        features = compareBlock.compute(candidatesBlock, pandasMem_df)
        features[col_to_block] = features[col_to_block].apply(lambda x: x*10)
        matchesRule = features[features[lst[:-1]].sum(axis=1) >= threshold]
        matchesRule_df = matchesRule.index.to_frame()
        logger.info(f"Rule {i}: {len(matchesRule)} matches found (threshold: {threshold})")
        if len(matchesRule_df) > 0:
            matchesAllRules_df = pd.concat([matchesAllRules_df, matchesRule_df])
            
    logger.info(f"RulesToCompare completed. Total matches across all rules: {len(matchesAllRules_df)}")
    return matchesAllRules_df

In [0]:
def RunLinking(sparkMem_df):
    logger.info("Entering RunLinking function")
    pandasMem_df = sparkMem_df.toPandas()
    pandasMem_df = pandasMem_df.set_index("UniqueRecord", drop=False)
    dfCombined = RulesToCompare(RulesAll, pandasMem_df)
    
    if dfCombined.empty:
        logger.info("NO MATCHES FOUND across any rules. Falling back to self-assignment path.")
        finalPandas_df = pandasMem_df.copy()
        finalPandas_df['MatchID'] = finalPandas_df['UniqueRecord']
        return spark.createDataFrame(finalPandas_df.astype(str))
        
    dfCombined.columns = [0, 1]
    dfMatchedColumnA = dfCombined.rename(columns={0:"A", 1:"B"})
    dfMatchedColumnB = dfCombined.rename(columns={0:"B", 1:"A"})
    dfMatched = pd.concat([dfMatchedColumnA, dfMatchedColumnB], ignore_index=True)
    dfMatched.drop_duplicates(inplace=True)
    
    matchesAll_df = pd.concat([
        dfMatched[["A", "B"]].rename(columns={"A": "Record", "B": "Match"}),
        dfMatched[["B", "A"]].rename(columns={"B": "Record", "A": "Match"})
    ]).reset_index(drop=True)
    
    matchesSame_df = matchesAll_df[["Record", "Record"]].copy()
    matchesSame_df.columns = ['Record', 'Match']
    
    matchesAll_df = pd.concat([matchesAll_df, matchesSame_df], ignore_index=True)
    matchesAll_df.drop_duplicates(inplace=True)
    
    distinctRow = dfMatched.groupby('A').head(1).drop('B', axis=1)
    matchesAll_df = pd.merge(matchesAll_df, distinctRow, how="left", left_on="Record", right_on="A").drop('A', axis=1)
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Record", right_index=True).rename(columns={"UniqueRecord":"RecordID"})
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Match", right_index=True).rename(columns={"UniqueRecord":"MatchID"})
    
    matched_df = matchesAll_df.groupby("RecordID") \
            .agg({"MatchID": lambda x: list(pd.unique(x))}) \
            .reset_index()
    matched_df["MatchID"] = matched_df["MatchID"].apply(lambda x: sorted(x))
    
    matchesAllModified = matchesAll_df.groupby("RecordID").head(1)
    drop_cols = ['MatchID', 'Match', 'Record']
    existing_drops = [c for c in drop_cols if c in matchesAllModified.columns]
    matchesAllModified = matchesAllModified.drop(columns=existing_drops)
    if 'index' in matchesAllModified.columns:
        matchesAllModified = matchesAllModified.drop(columns=['index'])
        
    newDFToMatch = pd.merge(matched_df, matchesAllModified, how="left", on="RecordID")
    finalPandas_df = pandasMem_df.reset_index(drop=True).merge(newDFToMatch, left_on="UniqueRecord", right_on="RecordID", how="left")
    finalPandas_df['MatchID'] = finalPandas_df['MatchID'].fillna(finalPandas_df['UniqueRecord'])
    return spark.createDataFrame(finalPandas_df.astype(str))

In [0]:
finalSQL = """
WITH BISPersonWithIdentifiers AS(
SELECT 
   UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,NPI,DEA,PayorID,ProviderID
  ,case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end as MatchID
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId ASC, RowNumber ASC) AS FirstPersonIdentifier
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId DESC, RowNumber DESC) AS CurrentPersonIdentifier
FROM BISCompletePersonTable
)
,ProviderPersonBridge AS(
SELECT 
   fp.UniqueRecord AS BISInternalPersonID
  ,CASE WHEN cp.CurrentPersonIdentifier = 1 THEN 1 ELSE 0 END AS IsCurrent
  ,cp.UniqueRecord,cp.FileLayoutID,cp.FileId,cp.RowNumber,cp.LastName,cp.FirstName,cp.NPI,cp.DEA,cp.PayorID,cp.ProviderID,cp.MatchID
FROM BISPersonWithIdentifiers cp
  LEFT JOIN BISPersonWithIdentifiers fp ON cp.MatchId = fp.MatchId AND fp.FirstPersonIdentifier = 1
)
,ProviderPersonBridge_CurrPlanMbr AS(
SELECT 
   BISInternalPersonID,IsCurrent,UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,NPI,DEA,PayorID,ProviderID
  ,ifnull(nullif(ProviderID,'None'),'') AS ProviderIdModified
  ,ifnull(nullif(NPI,'None'),'') AS NPIModified
  ,MatchID
  ,case when ifnull(ProviderID,'None')='None' then null when row_number() over(partition by ProviderID order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentProviderID
  ,case when ifnull(NPI,'None')='None' then null when row_number() over(partition by NPI order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentNPI
  ,case when BISInternalPersonID = UniqueRecord then 1 else 0 end AS IsOriginalProviderID 
FROM ProviderPersonBridge
)
,PUModPop AS(
SELECT *
    ,CASE WHEN ProviderIdModified <> '' THEN 1 ELSE 0 END AS IsProviderIdPopulated
    ,CASE WHEN NPIModified <> '' THEN 1 ELSE 0 END AS IsNPIPopulated
    ,concat(ProviderIdModified,'-',NPIModified) AS PMUP
FROM ProviderPersonBridge_CurrPlanMbr
)
,Final AS (
SELECT *
    ,CAST(COALESCE(IsCurrentProviderID,IsCurrentNPI) AS STRING) AS IsCurrentPMUP
FROM PUModPop
)
SELECT 
   BISInternalPersonID
  ,IsCurrent
  ,UniqueRecord
  ,CAST(FileLayoutID AS INT) AS FileLayoutID
  ,FileId
  ,LastName
  ,FirstName
  ,NPI
  ,DEA
  ,PayorID
  ,ProviderID
  ,sha2(concat_ws('|',IfNull(BISInternalPersonID,''),IfNull(IsCurrent,''),IfNull(UniqueRecord,''),IfNull(CAST(FileLayoutID AS STRING),''),IfNull(CAST(FileId AS STRING),''),IfNull(LastName,''),IfNull(FirstName,''),IfNull(NPI,''),IfNull(DEA,''),IfNull(PayorID,''),IfNull(ProviderID,''),IfNull(CAST(IsCurrentProviderID AS STRING),''),IfNull(CAST(IsCurrentNPI AS STRING),''),IfNull(CAST(IsOriginalProviderID AS STRING),''),IfNull(PMUP,''),IfNull(TRY_CAST(IsCurrentPMUP AS STRING),'')), 256) AS hashKey
  ,IsCurrentProviderID
  ,IsCurrentNPI
  ,IsOriginalProviderID
  ,PMUP
  ,TRY_CAST(IsCurrentPMUP AS INT) AS IsCurrentPMUP
FROM Final
"""

In [0]:
logger.info("MAIN EXECUTION STARTED")
print(f"Processing data from: {sourcePath}")

if table_exists(sourcePath):
    sparkMem_df = LoadSource(sourcePath)
    numRows = sparkMem_df.count()
    print(f"Found {numRows} records")
    
    if numRows == 0:
        print("No records to process")
        logger.info("No records to process - exiting")
    elif numRows == 1:
        print("Single record - skipping linkage, processing directly")
        convertedSpark_df = sparkMem_df.withColumn("MatchID", col("UniqueRecord"))
        convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
        
        convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
        temp_df = spark.sql(finalSQL)
        
        print(f"Writing to Silver table: {silverTable}")
        spark.sql(f"DROP TABLE IF EXISTS {silverTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(silverTable)
        print("Silver layer processing completed successfully!")
    else:
        print("Running record linkage...")
        convertedSpark_df = RunLinking(sparkMem_df)
        convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
        
        convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
        temp_df = spark.sql(finalSQL)
        
        print(f"Writing to Silver table: {silverTable}")
        spark.sql(f"DROP TABLE IF EXISTS {silverTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(silverTable)
        print("Silver layer processing completed successfully!")
else:
    print("Bronze source path does not exist. Skipping execution.")
    logger.info("Bronze source path missing - exiting")